# Comprehensive SiteX Data Engineering Pipeline

This notebook implements a fully normalized relational database schema for SiteX data. It transforms complex nested JSON into the following table structure:
- `place`: Core information (deduplicated)
- `categories_detailed`: All categories associated with a place
- `images`: Summary images
- `images_detailed`: Full list of images with titles
- `open_hours_detailed`: Daily schedule and intervals
- `owner_info`: Details about the business owner
- `popular_times_detailed`: Hourly popularity data by day
- `ratings_distribution`: 1-5 star review counts
- `reviews`: Primary review data
- `reviews_detailed`: Extended review metadata

In [ ]:
import sqlite3
import json
import os
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import ast
import re

# --- Helper Functions ---

def parse_time_string(time_str):
    if not isinstance(time_str, str):
        return None
    time_str = time_str.replace(' AM', ' AM').replace(' PM', ' PM').strip().replace('.', '')
    formats = ["%I:%M %p", "%I %p"]
    for fmt in formats:
        try:
            dt_object = datetime.strptime(time_str, fmt)
            return timedelta(hours=dt_object.hour, minutes=dt_object.minute)
        except ValueError:
            pass
    return None

def calculate_duration_from_intervals(intervals):
    total_duration_minutes = 0
    if not intervals or not isinstance(intervals, list):
        return 0.0
    for interval_str in intervals:
        if not isinstance(interval_str, str):
            continue
        interval_str = interval_str.replace('–', '-').replace('—', '-')
        if 'Open 24 hours' in interval_str or '24 hours' in interval_str.lower():
            total_duration_minutes += 24 * 60
            continue
        if 'Closed' in interval_str:
            continue
        parts = interval_str.split('-')
        if len(parts) != 2:
            continue
        try:
            start_td = parse_time_string(parts[0].strip())
            end_td = parse_time_string(parts[1].strip())
            if start_td is None or end_td is None:
                continue
            duration = end_td - start_td
            if duration < timedelta(0):
                duration += timedelta(days=1)
            total_duration_minutes += duration.total_seconds() / 60
        except ValueError:
            continue
    return total_duration_minutes / 60.0

def calculate_total_open_hours(open_hours_data):
    if not open_hours_data or not isinstance(open_hours_data, dict):
        return float('nan')
    daily_hours = {}
    for day, intervals in open_hours_data.items():
        daily_hours[day] = calculate_duration_from_intervals(intervals)
    total_sum_hours = sum(daily_hours.values())
    num_non_zero_days = sum(1 for h in daily_hours.values() if h > 0)
    if num_non_zero_days == 1:
        return total_sum_hours * 6
    return total_sum_hours

def clean_value(val):
    if isinstance(val, str):
        return val.replace('`', '').strip()
    return val

## Data Loading and Initial Deduplication

In [ ]:
json_file = 'all_cafe_jq.json'
with open(json_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

df_raw = pd.DataFrame(data)
print(f"Initial records: {len(df_raw)}")

# Remove duplicates
if 'place_id' in df_raw.columns:
    df_raw = df_raw.drop_duplicates(subset=['place_id']).reset_index(drop=True)
elif 'input_id' in df_raw.columns:
    df_raw = df_raw.drop_duplicates(subset=['input_id']).reset_index(drop=True)

print(f"Records after deduplication: {len(df_raw)}")

## Normalization and Relational Database Export

In [ ]:
db_file = 'cafes_comprehensive.db'
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# 1. Define Tables Schema
cursor.executescript("""
    DROP TABLE IF EXISTS place;
    DROP TABLE IF EXISTS categories_detailed;
    DROP TABLE IF EXISTS images;
    DROP TABLE IF EXISTS images_detailed;
    DROP TABLE IF EXISTS open_hours_detailed;
    DROP TABLE IF EXISTS owner_info;
    DROP TABLE IF EXISTS popular_times_detailed;
    DROP TABLE IF EXISTS ratings_distribution;
    DROP TABLE IF EXISTS reviews;
    DROP TABLE IF EXISTS reviews_detailed;

    CREATE TABLE place (
        place_id TEXT PRIMARY KEY,
        input_id TEXT,
        title TEXT,
        category TEXT,
        address TEXT,
        latitude REAL,
        longitude REAL,
        phone TEXT,
        plus_code TEXT,
        review_count INTEGER,
        review_rating REAL,
        price_range TEXT,
        total_open_hours REAL,
        city TEXT,
        borough TEXT,
        street TEXT,
        postal_code TEXT,
        timezone TEXT,
        status TEXT,
        description TEXT,
        link TEXT,
        thumbnail TEXT,
        menu_link TEXT
    );

    CREATE TABLE categories_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        category_name TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE images (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        image_url TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE images_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        title TEXT,
        image_url TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE open_hours_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        day TEXT,
        hours TEXT,
        duration_hours REAL,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE owner_info (
        place_id TEXT PRIMARY KEY,
        owner_id TEXT,
        owner_name TEXT,
        owner_link TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE popular_times_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        day TEXT,
        hour INTEGER,
        popularity INTEGER,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE ratings_distribution (
        place_id TEXT PRIMARY KEY,
        rating_1 INTEGER,
        rating_2 INTEGER,
        rating_3 INTEGER,
        rating_4 INTEGER,
        rating_5 INTEGER,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE reviews (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        reviewer_name TEXT,
        rating INTEGER,
        description TEXT,
        date TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE reviews_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        reviewer_name TEXT,
        rating INTEGER,
        description TEXT,
        date TEXT,
        profile_picture TEXT,
        images TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );
""")

# 2. Populate Tables
for _, row in df_raw.iterrows():
    pid = row.get('place_id') or row.get('input_id')
    
    # --- Place Table ---
    addr = row.get('complete_address', {})
    place_data = (
        pid, row.get('input_id'), row.get('title'), row.get('category'), 
        row.get('address'), row.get('latitude'), row.get('longtitude'),
        row.get('phone'), row.get('plus_code'), row.get('review_count'),
        row.get('review_rating'), row.get('price_range'), 
        calculate_total_open_hours(row.get('open_hours')),
        addr.get('city'), addr.get('borough'), addr.get('street'), addr.get('postal_code'),
        row.get('timezone'), row.get('status'), row.get('description'),
        clean_value(row.get('link')), clean_value(row.get('thumbnail')),
        clean_value(row.get('menu', {}).get('link')) if isinstance(row.get('menu'), dict) else None
    )
    cursor.execute("INSERT OR IGNORE INTO place VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", place_data)

    # --- Categories Detailed ---
    cats = row.get('categories', [])
    if isinstance(cats, list):
        for c in cats:
            cursor.execute("INSERT INTO categories_detailed (place_id, category_name) VALUES (?, ?)", (pid, c))

    # --- Images ---
    thumb = clean_value(row.get('thumbnail'))
    if thumb:
        cursor.execute("INSERT INTO images (place_id, image_url) VALUES (?, ?)", (pid, thumb))

    # --- Images Detailed ---
    imgs = row.get('images', [])
    if isinstance(imgs, list):
        for img in imgs:
            if isinstance(img, dict):
                cursor.execute("INSERT INTO images_detailed (place_id, title, image_url) VALUES (?, ?, ?)", 
                               (pid, img.get('title'), clean_value(img.get('image'))))

    # --- Open Hours Detailed ---
    oh = row.get('open_hours', {})
    if isinstance(oh, dict):
        for day, intervals in oh.items():
            duration = calculate_duration_from_intervals(intervals)
            cursor.execute("INSERT INTO open_hours_detailed (place_id, day, hours, duration_hours) VALUES (?, ?, ?, ?)",
                           (pid, day, ", ".join(intervals) if isinstance(intervals, list) else str(intervals), duration))

    # --- Owner Info ---
    owner = row.get('owner', {})
    if isinstance(owner, dict) and owner:
        cursor.execute("INSERT OR IGNORE INTO owner_info VALUES (?, ?, ?, ?)", 
                       (pid, owner.get('id'), owner.get('name'), clean_value(owner.get('link'))))

    # --- Popular Times Detailed ---
    pt = row.get('popular_times', {})
    if isinstance(pt, dict):
        for day, hours in pt.items():
            if isinstance(hours, dict):
                for h, pop in hours.items():
                    cursor.execute("INSERT INTO popular_times_detailed (place_id, day, hour, popularity) VALUES (?, ?, ?, ?)",
                                   (pid, day, int(h), int(pop)))

    # --- Ratings Distribution ---
    rd = row.get('reviews_per_rating', {})
    if isinstance(rd, dict) and rd:
        cursor.execute("INSERT OR IGNORE INTO ratings_distribution VALUES (?, ?, ?, ?, ?, ?)", 
                       (pid, rd.get('1', 0), rd.get('2', 0), rd.get('3', 0), rd.get('4', 0), rd.get('5', 0)))

    # --- Reviews ---
    revs = row.get('user_reviews', [])
    if isinstance(revs, list):
        for r in revs:
            if isinstance(r, dict):
                cursor.execute("INSERT INTO reviews (place_id, reviewer_name, rating, description, date) VALUES (?, ?, ?, ?, ?)",
                               (pid, r.get('Name'), r.get('Rating'), r.get('Description'), r.get('When')))

    # --- Reviews Detailed ---
    revs_ext = row.get('user_reviews_extended', [])
    combined_revs = (revs if isinstance(revs, list) else []) + (revs_ext if isinstance(revs_ext, list) else [])
    for r in combined_revs:
        if isinstance(r, dict):
            cursor.execute("INSERT INTO reviews_detailed (place_id, reviewer_name, rating, description, date, profile_picture, images) VALUES (?, ?, ?, ?, ?, ?, ?)",
                           (pid, r.get('Name'), r.get('Rating'), r.get('Description'), r.get('When'), 
                            clean_value(r.get('ProfilePicture')), str(r.get('Images', []))))

conn.commit()
conn.close()
print(f"Comprehensive database finalized at: {db_file}")

## Verification
List all tables and check data counts.

In [ ]:
conn = sqlite3.connect('cafes_comprehensive.db')
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';", conn)
print("Tables in Database:")
display(tables)

for t in tables['name']:
    count = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {t}", conn)['count'][0]
    print(f"- {t}: {count} records")
conn.close()